# Process StatsBomb data and generate datasets

This notebook loads data from the `data/statsbomb-open-data` directory, processes it using the `socceraction` library, and generates datasets for training supervised models.

In [ ]:
# !pip install socceraction
# !pip install tables

In [1]:
import warnings
import os
import pandas as pd

from socceraction.data.statsbomb import StatsBombLoader
import socceraction.spadl as spadl
import socceraction.vaep.features as features
import socceraction.vaep.labels as labels

warnings.filterwarnings('ignore', category=pd.io.pytables.PerformanceWarning)

## 1. Load StatsBomb data

Define the path to the `statsbomb-open-data` and use the `StatsBombLoader` from `socceraction` to load the data.

In [4]:
# Path to the StatsBomb open data
data_folder = 'data'

# Initialize the StatsBombLoader
SBL = StatsBombLoader(root=data_folder, getter='local')

# Load competitions, games, and events
df_competitions = SBL.competitions()
df_games = SBL.games(competition_id=43, season_id=3)

df_events = pd.concat(
    [SBL.events(gid) for gid in df_games.game_id],
    ignore_index=True
)

print(f"Loaded {len(df_competitions)} competitions")
print(f"Loaded {len(df_games)} games")
print(f"Loaded {len(df_events)} events")


/Users/antonio/PycharmProjects/StatsBomb-Open-Data/open-data/.venv/lib/python3.11/site-packages/socceraction/data/statsbomb/loader.py:337: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["under_pressure"] = eventsdf["under_pressure"].fillna(False).astype(bool)
/Users/antonio/PycharmProjects/StatsBomb-Open-Data/open-data/.venv/lib/python3.11/site-packages/socceraction/data/statsbomb/loader.py:338: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["counterpress"] = eventsdf["counterpress"].fillna(False).astype(bool)
/Users/antonio/PycharmProjec

Loaded 75 competitions
Loaded 64 games
Loaded 227849 events


/Users/antonio/PycharmProjects/StatsBomb-Open-Data/open-data/.venv/lib/python3.11/site-packages/socceraction/data/statsbomb/loader.py:337: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["under_pressure"] = eventsdf["under_pressure"].fillna(False).astype(bool)
/Users/antonio/PycharmProjects/StatsBomb-Open-Data/open-data/.venv/lib/python3.11/site-packages/socceraction/data/statsbomb/loader.py:338: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["counterpress"] = eventsdf["counterpress"].fillna(False).astype(bool)
/Users/antonio/PycharmProjec

## 2. Convert to SPADL format

The `socceraction` library uses the SPADL (Soccer Player Action Description Language) format. This step converts the StatsBomb events into SPADL actions.

In [7]:
df_games = SBL.games(competition_id=43, season_id=3)

all_actions = []

for _, game in df_games.iterrows():
    gid = game.game_id
    events = SBL.events(gid)
    home_team_id = game.home_team_id

    actions = spadl.statsbomb.convert_to_actions(events, home_team_id)
    all_actions.append(actions)

df_actions = pd.concat(all_actions, ignore_index=True)


/Users/antonio/PycharmProjects/StatsBomb-Open-Data/open-data/.venv/lib/python3.11/site-packages/socceraction/data/statsbomb/loader.py:337: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["under_pressure"] = eventsdf["under_pressure"].fillna(False).astype(bool)
/Users/antonio/PycharmProjects/StatsBomb-Open-Data/open-data/.venv/lib/python3.11/site-packages/socceraction/data/statsbomb/loader.py:338: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["counterpress"] = eventsdf["counterpress"].fillna(False).astype(bool)
/Users/antonio/PycharmProjec

## 3. Generate game state features

This section generates features to describe the game states. These features will be used to train the predictive models.

In [8]:
nb_prev_actions = 3

functions_features = [
    features.actiontype_onehot,
    features.bodypart_onehot,
    features.result_onehot,
    features.goalscore,
    features.startlocation,
    features.endlocation,
    features.movement,
    features.space_delta,
    features.startpolar,
    features.endpolar,
    features.team,
    features.time_delta
]

# Generate features for all games
with pd.HDFStore('spadl.h5', 'r') as spadl_store:
    df_actions = spadl_store['actions']
    games = spadl_store['games']
    
    dfs_features = []
    for _, game in games.iterrows():
        actions = df_actions[df_actions.game_id == game.game_id]
        gamestates = features.generate_gamestates(actions, nb_prev_actions=nb_prev_actions)
        
        dfs_gamestate_features = [fn(gamestates) for fn in functions_features]
        df_gamestate_features = pd.concat(dfs_gamestate_features, axis=1)
        
        df_gamestate_features['game_id'] = game.game_id
        dfs_features.append(df_gamestate_features)
        
df_features = pd.concat(dfs_features).reset_index(drop=True)

# Save features
if os.path.exists('features.h5'):
    os.remove('features.h5')
df_features.to_hdf('features.h5', key='features', mode='w', format='table')

print("Features generated and saved to features.h5")
df_features.head()

FileNotFoundError: ``/Users/antonio/PycharmProjects/StatsBomb-Open-Data/open-data/spadl.h5`` does not exist

## 4. Generate game state labels

This section generates the labels that capture the value of the game states. The labels are:
- `scores`: Did the team score in the near future?
- `concedes`: Did the team concede a goal in the near future?

In [ ]:
functions_labels = [
    labels.scores,
    labels.concedes
]

# Generate labels for all games
with pd.HDFStore('spadl.h5', 'r') as spadl_store:
    df_actions = spadl_store['actions']
    games = spadl_store['games']
    
    dfs_labels = []
    for _, game in games.iterrows():
        actions = df_actions[df_actions.game_id == game.game_id]
        
        dfs_game_labels = [fn(actions) for fn in functions_labels]
        df_game_labels = pd.concat(dfs_game_labels, axis=1)
        
        df_game_labels['game_id'] = game.game_id
        dfs_labels.append(df_game_labels)

df_labels = pd.concat(dfs_labels).reset_index(drop=True)

# Save labels
if os.path.exists('labels.h5'):
    os.remove('labels.h5')
df_labels.to_hdf('labels.h5', key='labels', mode='w', format='table')

print("Labels generated and saved to labels.h5")
df_labels.head()

## 5. Datasets ready for training

The `features.h5` and `labels.h5` files contain the datasets needed to train supervised models.

You can now use these files to train your models. For example, you can train a model to predict if a goal will be scored (`scores` label) or conceded (`concedes` label) based on the game state features.

In [ ]:
print("Features dataframe shape:", df_features.shape)
print("Labels dataframe shape:", df_labels.shape)